In [7]:
import os
import duckdb
from dotenv import load_dotenv

load_dotenv()

pg_host = os.getenv('PG_HOST')
pg_port = os.getenv('PG_PORT')
pg_dbname = os.getenv('PG_DBNAME')
pg_user = os.getenv('PG_USER')
pg_password = os.getenv('PG_PASSWORD')

duck_con = duckdb.connect("musicbrainz.duckdb")

duck_con.execute("""
INSTALL postgres;
LOAD postgres;
""")

duck_con.execute(f"""
ATTACH IF NOT EXISTS 'host={pg_host} port={pg_port} dbname={pg_dbname} user={pg_user} password={pg_password}'
AS mb_pg
(TYPE postgres, READ_ONLY, SCHEMA {pg_user});
""")

In [9]:
from pathlib import Path
import pandas as pd

ROOT = Path("..").resolve()

DUCKDB_PATH = ROOT / "musicbrainz.duckdb"
SQL_PATH = ROOT / "sql_features" / "queries" / "mb_artist_country_fast_duckdb_release.sql"


In [12]:
duck_con.execute("USE main;")

sql = SQL_PATH.read_text(encoding="utf-8-sig")
duck_con.execute(sql)



In [13]:
PARQUET_PATH = ROOT / "data" / "sql_feature_artist_country_fast.parquet"
duck_con.execute(f"""
COPY artist_country_fast
TO '{PARQUET_PATH}'
(FORMAT PARQUET, COMPRESSION ZSTD);
""")

In [14]:
df = duck_con.sql("""
SELECT *
FROM artist_country_fast
""").df()

In [15]:
duck_con.execute("USE main;")
duck_con.sql("""
DESCRIBE artist_country_fast;
""").show()

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ artist_id          │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ artist_name        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ area_id            │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ area_name          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ country_id         │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ area_is_missing    │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ country_id_imputed │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
└────────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘



In [16]:
duck_con.sql("""
SELECT
    COUNT(*)                                                        AS total_artists,
    SUM(area_is_missing::INT)                                       AS missing_area,
    ROUND(100.0 * SUM(area_is_missing::INT) / COUNT(*), 2)          AS pct_missing,
    SUM(
        CASE WHEN country_id_imputed IS NOT NULL
              AND area_is_missing THEN 1 ELSE 0 END
    )                                                               AS imputable    
FROM artist_country_fast;
""").show()

┌───────────────┬──────────────┬─────────────┬───────────┐
│ total_artists │ missing_area │ pct_missing │ imputable │
│     int64     │    int128    │   double    │  int128   │
├───────────────┼──────────────┼─────────────┼───────────┤
│       2864675 │      1779072 │        62.1 │    534207 │
└───────────────┴──────────────┴─────────────┴───────────┘



In [17]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2864675 entries, 0 to 2864674
Data columns (total 7 columns):
 #   Column              Dtype
---  ------              -----
 0   artist_id           int32
 1   artist_name         str  
 2   area_id             Int32
 3   area_name           str  
 4   country_id          Int32
 5   area_is_missing     bool 
 6   country_id_imputed  Int32
dtypes: Int32(3), bool(1), int32(1), str(2)
memory usage: 142.5 MB


In [18]:
df[(df['area_is_missing'] == True) & (df['country_id_imputed'] != '<NA>')]

,artist_id,artist_name,area_id,area_name,country_id,area_is_missing,country_id_imputed
887,1064707,Azar Swan,7020,NaN,7020,True,240
888,913339,Rideout,<NA>,NaN,<NA>,True,107
889,1878748,Justin Hartinger,<NA>,NaN,<NA>,True,240
890,1878750,Skela,<NA>,NaN,<NA>,True,240
891,1143939,Anderblast,<NA>,NaN,<NA>,True,240
...,...,...,...,...,...,...,...
2864641,2935414,Aleksandr Lizogub,<NA>,NaN,<NA>,True,240
2864657,3194329,Richard Akingbehin,326,NaN,326,True,221
2864668,3153484,GUNØ∞,2531,NaN,2531,True,113
2864670,1941845,Nep Sidhu,5076,NaN,5076,True,221


In [19]:
df[(df['area_is_missing'] == True) & (df.country_id_imputed.isna())].artist_id

1201       2148881
1202        913313
1203        913315
1204        913371
1205       1878747
            ...   
2861555    3193000
2861556    3187261
2861557    3187270
2861558    3188762
2861559    3196306
Name: artist_id, Length: 1244865, dtype: int32

In [20]:
duck_con.execute("USE main;")
duck_con.sql("""
DESCRIBE t_area_country;
""").show()

┌────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│      column_name       │ column_type │  null   │   key   │ default │  extra  │
│        varchar         │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ area_id                │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ area_name              │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ area_type              │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ country_area_id        │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ effective_country_code │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└────────────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘



In [21]:
df_artist_country_fast  = pd.read_parquet('../data/sql_feature_artist_country_fast.parquet')

In [22]:
df_artist_country_fast.info()

<class 'pandas.DataFrame'>
RangeIndex: 2864675 entries, 0 to 2864674
Data columns (total 7 columns):
 #   Column              Dtype  
---  ------              -----  
 0   artist_id           int32  
 1   artist_name         str    
 2   area_id             float64
 3   area_name           str    
 4   country_id          float64
 5   area_is_missing     bool   
 6   country_id_imputed  float64
dtypes: bool(1), float64(3), int32(1), str(2)
memory usage: 167.1 MB
